## 01_Raw_Ingestion

In [0]:
import requests
import json
from datetime import datetime, timedelta, timezone

In [0]:
BASE_URL = "https://hapi.fhir.org/baseR4"

RESOURCE_TYPES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

PAGE_SIZE = 100

print("Base URL", BASE_URL)
print("Resources", RESOURCE_TYPES)
print("page size", PAGE_SIZE)

Base URL https://hapi.fhir.org/baseR4
Resources ['Patient', 'Encounter', 'Observation', 'Condition']
page size 100


In [0]:
# FOR Inceremental Date Range

END_DATE = datetime.now(timezone.utc).date()
START_DATE = END_DATE - timedelta(days = 3)

print("Start Date", START_DATE)
print("End Date", END_DATE)

Start Date 2026-09-09
End Date 2026-09-12


In [0]:

#Test the API

url = f"{BASE_URL}/Patient"

params = {
 "_lastUpdated" : f"ge{START_DATE.isoformat()}",
 "_lastUpdated" : f"lt{END_DATE.isoformat()}",
 "_count" : PAGE_SIZE
}

response = requests.get(url, params= params, timeout=60)
print("HTTP Status", response.status_code)
print("Final Status", response.url)


HTTP Status 200
Final Status https://hapi.fhir.org/baseR4/Patient?_lastUpdated=lt2026-09-12&_count=100


In [0]:
#Convert API response to JSON

if response.status_code == 200:
    bundle = response.json()

    print("Resource Type:", bundle.get("resourceType"))
    print("Bundle Type:", bundle.get("type"))
    print("Total:", bundle.get("total"))
    print("Entries Returned:", len(bundle.get("entry", [])))
else:
    print("API Error:", response.text)

Resource Type: Bundle
Bundle Type: searchset
Total: None
Entries Returned: 100


In [0]:

# Inspect First Patient


entries = bundle.get("entry", [])

if entries:
    first_resource = entries[0].get("resource", {})

    print(json.dumps(first_resource, indent=2))
else:
    print("No records found.")

{
  "resourceType": "Patient",
  "id": "sindhu-syn-000006",
  "meta": {
    "versionId": "6",
    "lastUpdated": "2026-07-31T02:45:12.904-04:00",
    "source": "#C75WT8HzGJLAXRKr",
    "tag": [
      {
        "system": "https://sindhu-ecrf.local/tags",
        "code": "sindhu-synthetic-40"
      }
    ]
  },
  "identifier": [
    {
      "system": "https://sindhu-ecrf.local/synthetic-patient-id",
      "value": "SYN-000006"
    }
  ],
  "active": true,
  "name": [
    {
      "text": "Synthetic Patient SYN-000006",
      "family": "SYN-000006",
      "given": [
        "Synthetic"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "1965-01-01"
}


In [0]:
def fetch_fhir_resource(resource_type, start_date, end_date, page_size=100):

    # Keep FHIR resource name with correct capitalization
    resource_type = resource_type.capitalize()

    url = f"{BASE_URL}/{resource_type}"

    params = [
        ("_lastUpdated", f"ge{start_date.isoformat()}"),
        ("_lastUpdated", f"lt{end_date.isoformat()}"),
        ("_count", page_size)
    ]

    all_entries = []
    page_number = 1
    api_calls = []

    while url:

        print(f"Fetching {resource_type} - Page {page_number}")
        print(f"URL: {url}")

        response = requests.get(
            url,
            params=params if page_number == 1 else None,
            timeout=60
        )

        print("HTTP Status:", response.status_code)

        response.raise_for_status()

        bundle = response.json()

        extraction_timestamp = datetime.now(timezone.utc).isoformat()

        api_calls.append({
            "resource_type": resource_type,
            "page_number": page_number,
            "api_url_or_params": response.url,
            "extraction_timestamp": extraction_timestamp,
            "http_status": response.status_code
        })

        entries = bundle.get("entry", [])

        all_entries.extend(entries)

        print(f"Records returned: {len(entries)}")

        # Find next page
        next_url = None

        for link in bundle.get("link", []):

            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        url = next_url
        page_number += 1

    return all_entries, api_calls

In [0]:
# ==============================
# Test Patient Pagination
# ==============================

patient_entries, patient_api_logs = fetch_fhir_resource(
    resource_type="Patient",
    start_date=START_DATE,
    end_date=END_DATE,
    page_size=PAGE_SIZE
)

print("Total Patient Entries:", len(patient_entries))
print("Total API Calls:", len(patient_api_logs))

Fetching Patient - Page 1
URL: https://hapi.fhir.org/baseR4/Patient
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 2
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=100&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 3
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=200&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 4
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=300&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 5
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=400&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 6
URL: https://hapi.f

In [0]:
# ==============================
# Extract All FHIR Resources
# ==============================

all_resource_data = {}

for resource_type in RESOURCE_TYPES:

    entries, api_logs = fetch_fhir_resource(
        resource_type=resource_type,
        start_date=START_DATE,
        end_date=END_DATE,
        page_size=PAGE_SIZE
    )

    all_resource_data[resource_type] = {
        "entries": entries,
        "api_logs": api_logs
    }

    print(
        f"{resource_type}: "
        f"{len(entries)} records, "
        f"{len(api_logs)} API calls"
    )

Fetching Patient - Page 1
URL: https://hapi.fhir.org/baseR4/Patient
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 2
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=100&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 3
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=200&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 4
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=300&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 5
URL: https://hapi.fhir.org/baseR4?_getpages=1a517263-e4fb-453d-8175-2251f91c6f1e&_getpagesoffset=400&_count=100&_pretty=true&_bundletype=searchset
HTTP Status: 200
Records returned: 100
Fetching Patient - Page 6
URL: https://hapi.f

In [0]:
RAW_BASE_PATH = "/Volumes/workspace/fhir/raw_data"

print(RAW_BASE_PATH)

/Volumes/workspace/fhir/raw_data


In [0]:
page_number = 1

In [0]:
def fetch_and_save_fhir_resource(
    resource_type,
    start_date,
    end_date,
    page_size=100
):

    # Make sure resource name has correct capitalization
    resource_type = resource_type.capitalize()

    url = f"{BASE_URL}/{resource_type}"

    params = [
        ("_lastUpdated", f"ge{start_date.isoformat()}"),
        ("_lastUpdated", f"lt{end_date.isoformat()}"),
        ("_count", page_size)
    ]

    # Initialize variables
    page_number = 1
    total_records = 0
    api_logs = []

    extraction_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    while url:

        print(f"Fetching {resource_type} - Page {page_number}")

        response = requests.get(
            url,
            params=params if page_number == 1 else None,
            timeout=60
        )

        print("HTTP Status:", response.status_code)

        response.raise_for_status()

        bundle = response.json()

        # Timestamp for this API call
        extraction_timestamp = datetime.now(timezone.utc).isoformat()

        # Actual URL used
        api_url_or_params = response.url

        # Extract records from Bundle
        entries = bundle.get("entry", [])

        total_records += len(entries)

        # Raw file path
        raw_path = (
            f"{RAW_BASE_PATH}/"
            f"{resource_type}/"
            f"extraction_date={extraction_date}/"
            f"page_{page_number:03d}.json"
        )

        # Save complete FHIR Bundle as-is
        dbutils.fs.put(
            raw_path,
            json.dumps(bundle),
            overwrite=True
        )

        # Store API audit information
        api_logs.append({
            "resource_type": resource_type,
            "page_number": page_number,
            "extraction_timestamp": extraction_timestamp,
            "api_url_or_params": api_url_or_params,
            "http_status": response.status_code,
            "record_count": len(entries),
            "raw_path": raw_path
        })

        print(
            f"Saved {len(entries)} records → {raw_path}"
        )

        # Find next page
        next_url = None

        for link in bundle.get("link", []):

            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        # Move to next page
        url = next_url
        page_number += 1

    print(
        f"Completed {resource_type}: "
        f"{total_records} records"
    )

    return api_logs

In [0]:
raw_api_logs = []

for resource_type in RESOURCE_TYPES:

    logs = fetch_and_save_fhir_resource(
        resource_type=resource_type,
        start_date=START_DATE,
        end_date=END_DATE,
        page_size=PAGE_SIZE
    )

    raw_api_logs.extend(logs)

    print(
        f"{resource_type} completed - "
        f"{len(logs)} API calls"
    )

print("================================")
print("RAW INGESTION COMPLETED")
print("Total API calls:", len(raw_api_logs))
print("================================")

Fetching Patient - Page 1
HTTP Status: 200
Wrote 86405 bytes.
Saved 100 records → /Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_001.json
Fetching Patient - Page 2
HTTP Status: 200
Wrote 101208 bytes.
Saved 100 records → /Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_002.json
Fetching Patient - Page 3
HTTP Status: 200
Wrote 115547 bytes.
Saved 100 records → /Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_003.json
Fetching Patient - Page 4
HTTP Status: 200
Wrote 115501 bytes.
Saved 100 records → /Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_004.json
Fetching Patient - Page 5
HTTP Status: 200
Wrote 115714 bytes.
Saved 100 records → /Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_005.json
Fetching Patient - Page 6
HTTP Status: 200
Wrote 115586 bytes.
Saved 100 records → /Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_006.json
Fetching Pa

In [0]:
api_log_df = spark.createDataFrame(raw_api_logs)

display(api_log_df)

api_url_or_params,extraction_timestamp,http_status,page_number,raw_path,record_count,resource_type
https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-09-09&_lastUpdated=lt2026-09-12&_count=100,2026-09-12T09:44:50.503365+00:00,200,1,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_001.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=100&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:52.821965+00:00,200,2,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_002.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=200&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:53.751762+00:00,200,3,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_003.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=300&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:54.671032+00:00,200,4,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_004.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=400&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:55.539832+00:00,200,5,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_005.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=500&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:57.073577+00:00,200,6,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_006.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=600&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:57.971643+00:00,200,7,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_007.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=700&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:58.899806+00:00,200,8,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_008.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=800&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:59.751140+00:00,200,9,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_009.json,100,Patient
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=900&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:45:00.693100+00:00,200,10,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_010.json,100,Patient


In [0]:
from pyspark.sql.functions import current_timestamp

api_log_df = api_log_df.withColumn(
    "raw_save_timestamp",
    current_timestamp()
)

display(api_log_df)

api_url_or_params,extraction_timestamp,http_status,page_number,raw_path,record_count,resource_type,raw_save_timestamp
https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-09-09&_lastUpdated=lt2026-09-12&_count=100,2026-09-12T09:44:50.503365+00:00,200,1,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_001.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=100&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:52.821965+00:00,200,2,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_002.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=200&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:53.751762+00:00,200,3,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_003.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=300&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:54.671032+00:00,200,4,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_004.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=400&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:55.539832+00:00,200,5,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_005.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=500&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:57.073577+00:00,200,6,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_006.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=600&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:57.971643+00:00,200,7,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_007.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=700&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:58.899806+00:00,200,8,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_008.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=800&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:44:59.751140+00:00,200,9,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_009.json,100,Patient,2026-09-12T09:47:28.752Z
https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=900&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:45:00.693100+00:00,200,10,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_010.json,100,Patient,2026-09-12T09:47:28.752Z


In [0]:
metadata_path = f"{RAW_BASE_PATH}/_metadata"

dbutils.fs.mkdirs(metadata_path)

print("Metadata folder created:")
print(metadata_path)

Metadata folder created:
/Volumes/workspace/fhir/raw_data/_metadata


In [0]:
api_log_path = f"{RAW_BASE_PATH}/_metadata/api_calls"

(
    api_log_df
    .write
    .format("delta")
    .mode("append")
    .save(api_log_path)
)

print("API audit metadata saved successfully.")

API audit metadata saved successfully.


In [0]:
display(
    spark.read
    .format("delta")
    .load(api_log_path)
)

api_url_or_params,extraction_timestamp,http_status,page_number,raw_path,record_count,resource_type,raw_save_timestamp
https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-09-09&_lastUpdated=lt2026-09-12&_count=100,2026-09-12T09:18:22.704419+00:00,200,1,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_001.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=100&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:18:52.814240+00:00,200,2,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_002.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=200&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:18:54.010059+00:00,200,3,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_003.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=300&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:18:55.316037+00:00,200,4,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_004.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=400&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:18:56.811256+00:00,200,5,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_005.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=500&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:18:58.702827+00:00,200,6,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_006.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=600&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:18:59.856656+00:00,200,7,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_007.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=700&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:19:00.924071+00:00,200,8,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_008.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=800&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:19:02.088214+00:00,200,9,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_009.json,100,Patient,2026-09-12T09:21:41.045Z
https://hapi.fhir.org/baseR4?_getpages=ca185e27-3b2f-46a8-9435-625a4fbd238d&_getpagesoffset=900&_count=100&_pretty=true&_bundletype=searchset,2026-09-12T09:19:03.046188+00:00,200,10,/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_010.json,100,Patient,2026-09-12T09:21:41.045Z


In [0]:
from datetime import datetime, timezone

extraction_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

print("Extraction date:", extraction_date)

Extraction date: 2026-09-12


In [0]:
display(
    dbutils.fs.ls(RAW_BASE_PATH)
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/fhir/raw_data/Condition/,Condition/,0,1789206454896
dbfs:/Volumes/workspace/fhir/raw_data/Encounter/,Encounter/,0,1789206454896
dbfs:/Volumes/workspace/fhir/raw_data/Observation/,Observation/,0,1789206454896
dbfs:/Volumes/workspace/fhir/raw_data/Patient/,Patient/,0,1789206454896
dbfs:/Volumes/workspace/fhir/raw_data/_metadata/,_metadata/,0,1789206454896


In [0]:
display(
    dbutils.fs.ls(
        f"{RAW_BASE_PATH}/Patient/extraction_date={extraction_date}"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_001.json,page_001.json,86405,1789206292000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_002.json,page_002.json,101208,1789206294000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_003.json,page_003.json,115547,1789206294000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_004.json,page_004.json,115501,1789206295000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_005.json,page_005.json,115714,1789206296000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_006.json,page_006.json,115586,1789206298000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_007.json,page_007.json,115654,1789206299000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_008.json,page_008.json,115411,1789206300000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_009.json,page_009.json,115450,1789206300000
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/page_010.json,page_010.json,115549,1789206301000


In [0]:
sample_path = (
    f"{RAW_BASE_PATH}/Patient/"
    f"extraction_date={extraction_date}/"
    f"page_001.json"
)

sample_raw = dbutils.fs.head(sample_path, 5000)

print(sample_raw)

[Truncated to first 5000 bytes]
{"resourceType": "Bundle", "id": "10d19021-abd8-4a37-80e7-2dac4e667acb", "meta": {"lastUpdated": "2026-09-12T05:44:50.390-04:00"}, "type": "searchset", "link": [{"relation": "self", "url": "https://hapi.fhir.org/baseR4/Patient?_count=100&_lastUpdated=ge2026-09-09&_lastUpdated=lt2026-09-12"}, {"relation": "next", "url": "https://hapi.fhir.org/baseR4?_getpages=10d19021-abd8-4a37-80e7-2dac4e667acb&_getpagesoffset=100&_count=100&_pretty=true&_bundletype=searchset"}], "entry": [{"fullUrl": "https://hapi.fhir.org/baseR4/Patient/138279279", "resource": {"resourceType": "Patient", "id": "138279279", "meta": {"versionId": "1", "lastUpdated": "2026-09-08T06:19:15.018-04:00", "source": "#RRiDXLk0zBDRu2QW"}, "text": {"status": "generated", "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><div class=\"hapiHeaderText\">Nikolas <b>SOLOMON </b></div><table class=\"hapiPropertyTable\"><tbody><tr><td>Identifier</td><td>PJ1234567</td></tr><tr><td>Address</td><td><span>I

In [0]:
audit_df = (
    spark.read
    .format("delta")
    .load(api_log_path)
)

display(
    audit_df
    .groupBy("resource_type")
    .agg(
        {"record_count": "sum"}
    )
)

resource_type,sum(record_count)
Patient,12214
Observation,25919
Encounter,1867
Condition,16620


In [0]:
display(
    dbutils.fs.ls(
        f"{RAW_BASE_PATH}/Patient/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-10/,extraction_date=2026-09-10/,0,1789206458885
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-11/,extraction_date=2026-09-11/,0,1789206458885
dbfs:/Volumes/workspace/fhir/raw_data/Patient/extraction_date=2026-09-12/,extraction_date=2026-09-12/,0,1789206458885
